In [1]:
from predictions.prediction_tfidf_lsa import TfidfLsaModel
from config.global_config import TRAIN_ASPECTS

import pandas as pd

df = pd.read_csv('statics/datasets/training.csv')

df.head(3)

,time,rating,text,name,category,latitude,longitude,safety,cleanliness,infrastructure,nature,attractions,heritage,costs,other,created_at,id,lead_time,updated_at
0,1502577268143,5.0,"Union Square Park's playground, tho on the sma...",Union Square Park,Park||City park||Tourist attraction,40.735823499999995,-73.990521,notmentioned,notmentioned,positive,notmentioned,positive,notmentioned,notmentioned,negative,NaN,NaN,NaN,NaN
1,1555841530590,2.0,The supervisor was very rude to me. The pool i...,Liberty Pool,Public swimming pool||Outdoor swimming pool,40.701406,-73.784079,notmentioned,positive,neutral,notmentioned,notmentioned,notmentioned,notmentioned,negative,NaN,NaN,NaN,NaN
2,1582098542744,5.0,Lovely experience,Strawberry Fields,Tourist attraction||Historical landmark||Sceni...,40.7758392,-73.974766,notmentioned,notmentioned,notmentioned,notmentioned,notmentioned,notmentioned,notmentioned,positive,NaN,NaN,NaN,NaN


In [2]:
from predictions.prediction_fine_tuned import FineTunedModel

m = FineTunedModel(aspects=TRAIN_ASPECTS,local_model_path="saved_models/distilbert-base-uncased_absa.pt",)

for a in TRAIN_ASPECTS:
    df[a] = df[a].fillna("notmentioned")
    
print(df['text'][0])
print(m.predict(df['text'][0]))

/Users/bartoszbugla/projects/projekt-magisterski/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7890.70it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Union Square Park's playground, tho on the small side, is modern and filled with fun things for kids to play with. There's a giant orb/slide for climbing and sliding and a few other modern playground items in addition to slides and swings. Can get very crowded at peak times.
Using device: mps
{'safety': 'positive', 'cleanliness': 'positive', 'infrastructure': 'notmentioned', 'nature': 'positive', 'attractions': 'notmentioned', 'heritage': 'notmentioned', 'costs': 'notmentioned', 'other': 'neutral'}


In [ ]:

from sklearn.metrics import f1_score
from tqdm.auto import tqdm

from predictions.prediction_tfidf_lsa import TfidfLsaModel
from config.global_config import TRAIN_ASPECTS, SENTIMENT_LABELS

m = TfidfLsaModel(aspects=TRAIN_ASPECTS)

# TEST dla idf + lsa lae na tym samym zbiorze co zbiór treningowy
for a in TRAIN_ASPECTS:
    df[a] = df[a].fillna("notmentioned")

yt, yp = [], []

for _, r in tqdm(df.iterrows(), total=len(df), desc="TF-IDF + LSA", unit="row"):
    pred = m.predict(str(r["text"]))
    for a in TRAIN_ASPECTS:
        yt.append(r[a])
        yp.append(pred[a])
        
print(f1_score(yt, yp, labels=SENTIMENT_LABELS, average="macro", zero_division=0))


TF-IDF + LSA: 100%|██████████| 4251/4251 [00:03<00:00, 1108.81row/s]

0.6369026567336583
